In [0]:
# Databricks notebook source
import dlt
from pyspark.sql.functions import col, concat, lit, lower, monotonically_increasing_id, regexp_replace, trim, when
from pyspark.sql.types import DoubleType, IntegerType

TOP_DIR = "/Volumes/prd_mega/sboost4/vboost4"
WORKSPACE_DIR = f"{TOP_DIR}/Workspace"
COUNTRY = "Peru"
COUNTRY_MICRODATA_DIR = f"{WORKSPACE_DIR}/microdata_csv/{COUNTRY}"

CSV_READ_OPTIONS = {
    "header": "true",
    "multiline": "true",
    "quote": '"',
    "escape": '"',
}

STRING_COLUMNS = [
    "admin1",
    "econ1",
    "econ2",
    "econ3",
    "econ4",
    "function1",
    "function2",
    "function3",
    "source_fin1",
    "source_fin2",
]


def clean_label(column_name):
    return trim(regexp_replace(col(column_name).cast("string"), r"\s+", " "))


def strip_leading_code(column_name):
    return trim(regexp_replace(col(column_name), r"^[0-9A-Z]+\s+", ""))


def is_irrigation():
    return (
        (col("function1") == "10 AGROPECUARIA")
        & (
            (col("function2") == "025 RIEGO")
            | ((col("function2") == "009 CIENCIA Y TECNOLOGIA") & (col("function3") == "0050 INFRAESTRUCTURA DE RIEGO"))
            | ((col("function2") == "023 AGRARIO") & col("function3").isin("0050 INFRAESTRUCTURA DE RIEGO", "0051 RIEGO TECNIFICADO"))
            | ((col("function2") == "027 ACUICULTURA") & (col("function3") == "0050 INFRAESTRUCTURA DE RIEGO"))
        )
    )


def is_road():
    # The source CCI formulas double-counted urban roads by adding function3 to
    # broader road/transport terms. In gold, a matching line receives one label.
    return (
        col("function2").isin("52 TRANSPORTE TERRESTRE", "033 TRANSPORTE TERRESTRE")
        | col("function3").isin("157 VIAS URBANAS", "0074 VIAS URBANAS")
    )


def is_economic_affairs_function():
    return col("function1").isin(
        "04 AGRARIA",
        "07 TRABAJO",
        "08 COMERCIO",
        "09 TURISMO",
        "10 AGROPECUARIA",
        "10 ENERGIA Y RECURSOS MINERALES",
        "11 PESCA",
        "12 ENERGIA",
        "13 MINERIA",
        "14 INDUSTRIA",
        "15 TRANSPORTE",
        "16 COMUNICACIONES",
    )


@dlt.expect_or_drop("year_not_null", "year IS NOT NULL")
@dlt.table(name="per_boost_bronze")
def boost_bronze():
    df = (
        spark.read.format("csv")
        .options(**CSV_READ_OPTIONS)
        .option("inferSchema", "true")
        .load(f"{COUNTRY_MICRODATA_DIR}/Raw.csv")
    )

    for column_name in STRING_COLUMNS:
        df = df.withColumn(column_name, clean_label(column_name))

    return (
        df.withColumn("year", col("year").cast(IntegerType()))
        .withColumn("approved", col("monto_pia").cast(DoubleType()))
        .withColumn("executed", col("monto_devengado").cast(DoubleType()))
        .withColumn("id", concat(lit("per_"), monotonically_increasing_id()))
    )


@dlt.table(name="per_boost_silver")
def boost_silver():
    return (
        dlt.read("per_boost_bronze")
        .filter(~((col("econ3") == "81 AMORTIZACION DE LA DEUDA") | col("econ3").startswith("71")))
        .withColumn("admin0", when(col("admin1") == "1 GOBIERNO NACIONAL", lit("Central")).otherwise(lit("Regional")))
        .withColumn(
            "admin1_tmp",
            when(col("admin0") == "Central", lit("Central Scope")).otherwise(strip_leading_code("admin1")),
        )
        .withColumn("admin2_tmp", strip_leading_code("admin1"))
        .withColumn("geo1", when(col("admin0") == "Central", lit("Central Scope")).otherwise(col("admin1_tmp")))
        .withColumn("is_foreign", ~col("source_fin1").startswith("1 RECURSOS ORDINARIOS"))
        .withColumn("is_pension_contribution", (col("econ4") == "11 OBLIGACIONES DEL EMPLEADOR") | (col("econ3") == "13 CONTRIBUCIONES A LA SEGURIDAD SOCIAL"))
        .withColumn("is_wage", (col("econ2") == "1 PERSONAL Y OBLIGACIONES SOCIALES") & ~col("is_pension_contribution"))
        .withColumn("is_capital", col("econ1") == "6 GASTOS DE CAPITAL")
        .withColumn("is_goods_services", col("econ2") == "3 BIENES Y SERVICIOS")
        .withColumn("is_subsidy", col("econ3") == "51 SUBSIDIOS")
        .withColumn("is_pension", col("econ4") == "14 PENSIONES")
        .withColumn("is_interest", col("econ2") == "78 INTERESES Y CARGOS DE LA DEUDA")
        .withColumn("is_debt_repayment", col("econ2") == "79 AMORTIZACION DE LA DEUDA")
        .withColumn(
            "is_social_assistance",
            (col("function1") == "05 ASISTENCIA Y PREVISION SOCIAL")
            & ~col("is_wage")
            & ~col("is_capital")
            & ~col("is_goods_services")
            & ~col("is_subsidy")
            & ~col("is_pension"),
        )
        .withColumn(
            "func_sub",
            when(col("function1") == "02 JUSTICIA", lit("Judiciary"))
            .when(is_irrigation(), lit("Irrigation"))
            .when(is_road(), lit("Roads"))
            .when(col("function2").isin("53 TRANSPORTE FERROVIARIO", "034 TRANSPORTE FERROVIARIO"), lit("Railroads"))
            .when(col("function2").isin("035 TRANSPORTE HIDROVIARIO"), lit("Water Transport"))
            .when(col("function2").isin("51 TRANSPORTE AEREO", "032 TRANSPORTE AEREO"), lit("Air Transport"))
            .when(col("function1").isin("04 AGRARIA", "10 AGROPECUARIA", "11 PESCA"), lit("Agriculture"))
            .when(col("function1").isin("10 ENERGIA Y RECURSOS MINERALES", "12 ENERGIA"), lit("Energy"))
            .when(col("function1") == "06 COMUNICACIONES", lit("Telecoms"))
            .when((col("function1").isin("14 SALUD Y SANEAMIENTO", "20 SALUD")) & ~col("function2").isin("47 SANEAMIENTO"), lit("Health"))
            .when(col("function1").isin("09 EDUCACION Y CULTURA", "22 EDUCACION"), lit("Education"))
        )
        .withColumn(
            "func",
            when(col("function1").isin("04 DEFENSA Y SEGURIDAD NACIONAL", "05 ORDEN PUBLICO Y SEGURIDAD"), lit("Defence"))
            .when(col("func_sub") == "Judiciary", lit("Public order and safety"))
            .when(is_economic_affairs_function() | col("func_sub").isin("Irrigation", "Roads", "Railroads", "Water Transport", "Air Transport", "Agriculture", "Energy", "Telecoms"), lit("Economic affairs"))
            .when(col("function1").isin("17 MEDIO AMBIENTE"), lit("Environmental protection"))
            .when(col("function1").isin("14 SALUD Y SANEAMIENTO", "20 SALUD"), lit("Health"))
            .when(col("function1").isin("09 EDUCACION Y CULTURA", "22 EDUCACION"), lit("Education"))
            .when(col("function1") == "05 ASISTENCIA Y PREVISION SOCIAL", lit("Social protection"))
            .otherwise(lit("General public services"))
        )
        .withColumn(
            "econ_sub",
            when(col("is_pension_contribution"), lit("Social Benefits (pension contributions)"))
            .when(col("is_pension"), lit("Pensions"))
            .when(col("is_social_assistance"), lit("Social Assistance"))
            .when(col("econ4") == "3202 SERVICIOS BASICOS, COMUNICACIONES, PUBLICIDAD Y DIFUSION", lit("Basic Services"))
            .when(col("econ4").isin("3207 SERVICIOS PROFESIONALES Y TECNICOS", "3208 CONTRATO ADMINISTRATIVO DE SERVICIOS"), lit("Employment Contracts"))
            .when(col("econ4") == "3204 SERVICIO DE MANTENIMIENTO, ACONDICIONAMIENTO Y REPARACIONES", lit("Recurrent Maintenance"))
            .when(col("is_subsidy"), lit("Subsidies to Production"))
            .when(col("is_capital") & col("is_foreign"), lit("Capital Expenditure (foreign spending)"))
        )
        .withColumn(
            "econ",
            when(col("is_wage"), lit("Wage bill"))
            .when(col("is_capital"), lit("Capital expenditures"))
            .when(col("is_goods_services"), lit("Goods and services"))
            .when(col("is_subsidy"), lit("Subsidies"))
            .when(col("is_pension_contribution") | col("is_pension") | col("is_social_assistance"), lit("Social benefits"))
            .when(col("is_interest"), lit("Interest on debt"))
            .when(col("is_debt_repayment"), lit("Debt repayment"))
            .otherwise(lit("Other expenses"))
        )
    )


@dlt.table(name="per_boost_gold")
def boost_gold():
    return (
        dlt.read("per_boost_silver")
        .withColumn("country_name", lit(COUNTRY))
        .select(
            "country_name",
            "year",
            "admin0",
            col("admin1_tmp").alias("admin1"),
            col("admin2_tmp").alias("admin2"),
            "geo1",
            "func",
            "func_sub",
            "econ",
            "econ_sub",
            "is_foreign",
            "approved",
            lit(None).cast(DoubleType()).alias("revised"),
            "executed",
        )
    )
